In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
def load_features(file_path, meta_cols=None, label_col='label'):
    if meta_cols is None:
        meta_cols = []

    df = pd.read_csv(file_path)
    X = df.drop(columns=meta_cols + [label_col]).values

    # 1. Initialize LabelEncoder to handle the string labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(df[label_col].values)

    # Store the classes for the evaluation report
    target_names = le.classes_

    return X, y_encoded, target_names

def preprocess_features(X):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled

In [3]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

def run_classification(embedding, y, target_names=None, models=None, projector=None):
    """
    embedding: (n_samples, k) dense
    y: labels
    projector: optional function to transform embedding → P @ embedding
    """
    if target_names is not None:
        target_names = list(target_names)

    if models is None:
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.svm import SVC
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.tree import DecisionTreeClassifier

        models = {
            "Logistic Regression": LogisticRegression(max_iter=5000, solver="saga", random_state=42),
            "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
            "SVM":                 SVC(kernel="rbf", probability=True, random_state=42),
            "KNN":                 KNeighborsClassifier(n_neighbors=5),
            "Decision Tree":       DecisionTreeClassifier(random_state=42),
        }

    # Apply projector if provided
    if projector is not None:
        X_class = projector(embedding)  # P @ embedding, shape (n_samples, k)
    else:
        X_class = embedding

    results = {}
    for name, clf in models.items():
        y_pred = cross_val_predict(clf, X_class, y, cv=10)
        report = classification_report(y, y_pred, target_names=target_names, output_dict=True, zero_division=0)

        results[name] = {
            "classes": [
                {
                    "name":      cls,
                    "precision": round(report[cls]["precision"], 4),
                    "recall":    round(report[cls]["recall"], 4),
                    "f1":        round(report[cls]["f1-score"], 4),
                    "support":   int(report[cls]["support"]),
                }
                for cls in (target_names or sorted(set(str(c) for c in y)))
            ],
            "accuracy":        round(report["accuracy"], 4),
            "macro_f1":        round(report["macro avg"]["f1-score"], 4),
            "macro_precision": round(report["macro avg"]["precision"], 4),
            "macro_recall":    round(report["macro avg"]["recall"], 4),
        }

    return results

In [4]:
import pandas as pd
from IPython.display import display

def display_classification_results(results, display_summary=True):
    """
    Displays classification results in clean, research-level tables:
    1. Per-class metrics for each model
    2. Macro-averaged summary across all models, ranked by Macro F1
    """

    # ── 1. Per-class table ───────────────────────────────────────────────
    rows = []
    for model, data in results.items():
        for cls in data["classes"]:
            rows.append({
                "Model": model,
                "Class": cls["name"],
                "Precision": cls["precision"],
                "Recall": cls["recall"],
                "F1": cls["f1"],
                "Support": cls["support"],
            })
    df_class = pd.DataFrame(rows)

    # ── 2. Macro summary table ───────────────────────────────────────────
    summary_rows = []
    for model, data in results.items():
        summary_rows.append({
            "Model": model,
            "Accuracy": data["accuracy"],
            "Macro Precision": data["macro_precision"],
            "Macro Recall": data["macro_recall"],
            "Macro F1": data["macro_f1"],
        })
    df_summary = pd.DataFrame(summary_rows).sort_values("Macro F1", ascending=False)

    # ── Styling function ────────────────────────────────────────────────
    def style_table(df, metric_cols):
        return (
            df.style
            .format({c: "{:.3f}" for c in metric_cols})
            .highlight_max(subset=metric_cols, props="font-weight: bold;")
            .set_properties(**{"text-align": "center", "font-family": "Arial", "font-size": "12px"})
            .set_table_styles([
                {"selector": "thead th", "props": [("font-weight", "bold"), ("text-align", "center")]},
                {"selector": "tbody td", "props": [("padding", "4px 8px")]},
            ])
        )

    # ── Display ─────────────────────────────────────────────────────────

    if not display_summary:
        display(df_class.style.format({
            "Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"
        }).set_properties(**{"text-align": "center", "font-family": "Arial"}).set_caption("Per-class metrics"))

    display(style_table(df_summary, ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]).set_caption("Macro summary — ranked by Macro F1"))

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np


def run_cv_inference_attack(
    X,
    y,
    sensitive_class,
    cv          : int  = 10,
    random_state: int  = 42,
    stage_name  : str  = "",
    verbose     : bool = True,
) -> pd.DataFrame:
    """
    Attribute Inference Attack evaluated with stratified k-fold CV.

    The adversary is given the embedding X and tries to predict binary
    membership in `sensitive_class`.  Three metrics are reported:

        Adversary Accuracy  — raw accuracy of the attack model
        Random Baseline     — majority-class rate (zero-effort adversary)
        Privacy Gain        — baseline − adv_acc
                              > 0  : adversary WORSE than random  ✓ private
                              < 0  : adversary BETTER than random ✗ leaked
        Normalized Gain     — (adv_acc − baseline) / (1 − baseline)
                              0.0  : no advantage above random
                              1.0  : perfect attack
        AUC                 — AUROC of the adversary (threshold-free)

    Fixes vs original
    -----------------
    - Removed class_weight='balanced': this artificially boosts adversary
      recall on minority class, making attack look stronger than it is.
      An honest adversary optimises for accuracy, not recall.
    - Replaced plain cv=10 with StratifiedKFold to guarantee class
      balance in every fold, especially important when sensitive_class
      is a minority.
    - Added AUC and normalized gain for richer reporting.
    - Added stage_name for easy comparison across pipeline stages.
    """

    # ── 1. Binary target ──────────────────────────────────────────────
    y_adv    = (y == sensitive_class).astype(int)
    pos_rate = float(y_adv.mean())
    baseline = max(pos_rate, 1.0 - pos_rate)   # majority-class accuracy

    # ── 2. Adversary models ───────────────────────────────────────────
    # No class_weight — honest adversary maximises accuracy, not recall
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    adversaries = {
        "Logistic Regression (LR)": LogisticRegression(
            max_iter=2000, solver="saga", random_state=random_state, class_weight="balanced",
        ),
    }

    # ── 3. Evaluate each adversary ────────────────────────────────────
    rows = {}
    for name, clf in adversaries.items():

        # Hard predictions for accuracy
        y_pred  = cross_val_predict(
            clf, X, y_adv, cv=cv_splitter, method="predict"
        )

        # Soft predictions for AUC
        y_proba = cross_val_predict(
            clf, X, y_adv, cv=cv_splitter, method="predict_proba"
        )[:, 1]

        adv_acc  = float(accuracy_score(y_adv, y_pred))
        adv_f1   = float(f1_score(y_adv, y_pred, zero_division=0))
        adv_auc  = float(roc_auc_score(y_adv, y_proba))

        priv_gain  = baseline - adv_acc
        # Normalized: how much of the exploitable gap does adversary capture?
        # 0 = no advantage, 1 = perfect attack
        norm_gain  = (adv_acc - baseline) / (1.0 - baseline + 1e-10)

        rows[name] = {
            "Adv Accuracy"   : round(adv_acc,   4),
            "Random Baseline": round(baseline,   4),
            "Privacy Gain"   : round(priv_gain,  4),   # + good, - bad
            "Norm Gain"      : round(norm_gain,  4),   # lower is better
            "Adv F1"         : round(adv_f1,     4),
            "Adv AUC"        : round(adv_auc,    4),
        }

    # ── 4. Display ────────────────────────────────────────────────────
    df = pd.DataFrame(rows).T

    if verbose:
        header = f"Inference Attack — {stage_name}" if stage_name else "Inference Attack"
        print(f"\n{'='*62}")
        print(f"  {header}")
        print(f"{'='*62}")
        print(f"  Sensitive class : {sensitive_class}")
        print(f"  Class balance   : {pos_rate:.1%} positive  /  {1-pos_rate:.1%} negative")
        print(f"  Random baseline : {baseline:.4f}")
        print()

        col_w = 26
        print(f"  {'Adversary':<{col_w}} {'Acc':>6}  {'Gain':>7}  {'NormG':>6}  {'AUC':>6}  {'Status'}")
        print(f"  {'-'*62}")
        for name, r in rows.items():
            status = "✓ private" if r["Privacy Gain"] > 0 else "✗ leaked"
            print(
                f"  {name:<{col_w}}"
                f" {r['Adv Accuracy']:>6.4f}"
                f" {r['Privacy Gain']:>+7.4f}"
                f" {r['Norm Gain']:>6.4f}"
                f" {r['Adv AUC']:>6.4f}"
                f"  {status}"
            )
        print()

    return df

# Loading Feature Set & Preprocessing

In [6]:
x, y, target_names = load_features(
    file_path="labeled_features.csv",
    # meta_cols=['file_id', 'fault_size', 'load'],
    label_col='label'
)
X_scaled = preprocess_features(x)

In [ ]:
from specral_privacy_pipeline import DPLaplacianEigenmaps, SpectralGapNoise

model = DPLaplacianEigenmaps(
    n_neighbors=10,
    n_components=4,
    noise_mechanism=SpectralGapNoise(epsilon=1.0)
)

results = model.fit_transform(X_scaled)

V = results["embedding_clean"]            # The clean eigenvectors (n, k)
V_noisy = results["embedding_noisy"]        # V + delta_V

# State 1: Raw Feature Set
features_raw = X_scaled

# State 2: Clean Embedding (Standard LE)
# Usually, downstream tasks just use the rows of V as the new features
features_clean_embed = V

# State 3: Perturbed Embedding (Theorem 2)
# The noisy eigenvectors directly
features_noisy_embed = V_noisy

# State 4: Perturbed Eigenprojection (Theorem 3 applied to Raw Data)
# Projecting the raw n x d data into the perturbed n x n subspace
# This results in an n x d matrix, smoothed by the DP spectral graph
features_noisy_projection = results["embedding_projector"]

In [10]:
print("Raw features shape:", features_raw.shape)
print("clean embedding shape:", features_clean_embed.shape)
print("embedding noise shape:", features_noisy_embed.shape)
print("projected features shape:", features_noisy_projection.shape)

Raw features shape: (8767, 5)
clean embedding shape: (8767, 4)
embedding noise shape: (8767, 4)
projected features shape: (8767, 4)


# Setting Sensitive Class

In [9]:
sensitive_class_name = "B"
sensitive_label = np.where(target_names == sensitive_class_name)[0][0]

# Test Performance on Base data

In [12]:
model_results = run_classification(features_raw, y, target_names=target_names)
display_classification_results(model_results)
run_cv_inference_attack(features_raw, y, sensitive_class=sensitive_label)


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.787,0.794,0.800,0.796
4,Decision Tree,0.772,0.775,0.785,0.779
3,KNN,0.768,0.775,0.776,0.775
2,SVM,0.690,0.669,0.699,0.670
0,Logistic Regression,0.687,0.679,0.686,0.664



=== Inference Attack Results ===
                          Adversary Accuracy  Random Baseline  Privacy Gain
Logistic Regression (LR)              0.7022            0.784        0.0818
Random Forest (RF)                    0.8895            0.784       -0.1055


# Test Performance on Embedding

In [13]:
model_results = run_classification(features_clean_embed, y, target_names=target_names)
display_classification_results(model_results)
run_cv_inference_attack(features_noisy_embed, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.671,0.666,0.687,0.675
4,Decision Tree,0.659,0.661,0.673,0.666
3,KNN,0.660,0.655,0.678,0.665
2,SVM,0.678,0.615,0.672,0.622
0,Logistic Regression,0.470,0.585,0.368,0.333



=== Inference Attack Results ===
                          Adversary Accuracy  Random Baseline  Privacy Gain
Logistic Regression (LR)               0.819            0.784        -0.035
Random Forest (RF)                     0.863            0.784        -0.079


# Test Performance on Noisy Embedding

In [14]:
model_results = run_classification(features_noisy_embed, y, target_names=target_names)
display_classification_results(model_results)
run_cv_inference_attack(features_noisy_embed, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.666,0.660,0.682,0.670
4,Decision Tree,0.655,0.659,0.668,0.663
3,KNN,0.634,0.626,0.651,0.637
2,SVM,0.675,0.612,0.666,0.617
0,Logistic Regression,0.632,0.523,0.622,0.568



=== Inference Attack Results ===
                          Adversary Accuracy  Random Baseline  Privacy Gain
Logistic Regression (LR)              0.8190            0.784       -0.0350
Random Forest (RF)                    0.8626            0.784       -0.0786


# Running Classification on the Projector

In [16]:
classification_results = run_classification(
    embedding=features_noisy_projection,
    y=y,
    target_names=target_names,
)
display_classification_results(classification_results)
run_cv_inference_attack(features_noisy_projection, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.666,0.660,0.682,0.670
4,Decision Tree,0.655,0.659,0.668,0.663
3,KNN,0.634,0.626,0.651,0.637
2,SVM,0.675,0.612,0.666,0.617
0,Logistic Regression,0.632,0.523,0.622,0.568



=== Inference Attack Results ===
                          Adversary Accuracy  Random Baseline  Privacy Gain
Logistic Regression (LR)              0.8190            0.784       -0.0350
Random Forest (RF)                    0.8627            0.784       -0.0787


In [6]:
from specral_privacy_pipeline import DPLaplacianEigenmaps, PPSPLaplacianNoise, SpectralGapNoise

x, y, target_names = load_features(
    file_path="features_raw_0_overlap.csv",
    meta_cols=['file_id', 'fault_size', 'load'],
    label_col='label'
)

model = DPLaplacianEigenmaps(
    n_neighbors=7,
    n_components=3,
    noise_mechanism=PPSPLaplacianNoise(
        epsilon      = .5,
        y_train      = y,        # training labels only
        n_train      = len(y),
        delta_frob   = 0.3,
        random_state = 42,
    )
    # noise_mechanism=SpectralGapNoise(epsilon=1.0)
)

results = model.fit_transform(x)

V = results["embedding_clean"]            # The clean eigenvectors (n, k)
V_noisy = results["embedding_noisy"]        # V + delta_V

# State 1: Raw Feature Set
features_raw = x

# State 2: Clean Embedding (Standard LE)
# Usually, downstream tasks just use the rows of V as the new features
features_clean_embed = V

# State 3: Perturbed Embedding (Theorem 2)
# The noisy eigenvectors directly
features_noisy_embed = V_noisy

# State 4: Perturbed Eigenprojection (Theorem 3 applied to Raw Data)
# Projecting the raw n x d data into the perturbed n x n subspace
# This results in an n x d matrix, smoothed by the DP spectral graph
features_noisy_projection = results["embedding_projector"]

/home/zoso/Codes/privacy-pipeline/specral_privacy_pipeline.py:159: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  M = np.divide(1.0, diff, where=np.abs(diff) > tol)
/home/zoso/Codes/privacy-pipeline/specral_privacy_pipeline.py:174: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  M = np.divide(1.0, diff, where=np.abs(diff) > tol)


In [8]:
print("Clean embedding shape:", features_clean_embed.shape)
print("Noisy embedding shape:", features_noisy_embed.shape)
print("Noisy projection shape:", features_noisy_projection.shape)

Clean embedding shape: (8767, 3)
Noisy embedding shape: (8767, 3)
Noisy projection shape: (8767, 3)


In [9]:
model_results = run_classification(features_noisy_embed, y, target_names=target_names)
display_classification_results(model_results)
run_cv_inference_attack(features_noisy_embed, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
3,KNN,0.866,0.873,0.875,0.874
1,Random Forest,0.864,0.871,0.870,0.871
4,Decision Tree,0.843,0.849,0.850,0.849
2,SVM,0.748,0.762,0.770,0.755
0,Logistic Regression,0.659,0.846,0.610,0.606


NameError: name 'sensitive_label' is not defined

In [18]:
classification_results = run_classification(
    embedding=features_noisy_projection,
    y=y,
    target_names=target_names,
)
display_classification_results(classification_results)
run_cv_inference_attack(features_noisy_projection, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.819,0.827,0.832,0.829
3,KNN,0.796,0.805,0.814,0.809
4,Decision Tree,0.787,0.797,0.799,0.798
2,SVM,0.667,0.819,0.623,0.625
0,Logistic Regression,0.568,0.490,0.527,0.481



  Inference Attack
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  Status
  --------------------------------------------------------------
  Logistic Regression (LR)   0.8305 -0.0465 0.2154 0.7265  ✗ leaked
  Random Forest (RF)         0.9170 -0.1330 0.6156 0.9580  ✗ leaked



,Adv Accuracy,Random Baseline,Privacy Gain,Norm Gain,Adv F1,Adv AUC
Logistic Regression (LR),0.8305,0.784,-0.0465,0.2154,0.3751,0.7265
Random Forest (RF),0.9170,0.784,-0.1330,0.6156,0.8043,0.9580


In [25]:
class EmbeddingPerturbation:
    """
    MI-weighted + Gaussian output perturbation on the clean embedding V.

    Core fix vs previous version
    ----------------------------
    Noise scale is now anchored to the ACTUAL coordinate range of each
    eigenvector column, not a fixed formula borrowed from feature space.

    For eigenvectors:  entries are O(1/√n),  std per column ≈ 1/√n
    Correct scale:     noise_std_i = (w_i * std_i * amplifier) / epsilon

    This ensures the signal-to-noise ratio is controlled, not the
    absolute noise magnitude.
    """

    def __init__(
        self,
        epsilon     : float,
        y_train     : np.ndarray,
        n_train     : int,
        delta       : float = 1e-5,
        snr_target  : float = 2.0,    # signal / noise ratio target
        min_weight  : float = 0.05,
        random_state: int   = 42,
    ):
        self.epsilon     = epsilon
        self.delta       = delta
        self.y_train     = y_train
        self.n_train     = n_train
        self.snr_target  = snr_target   # higher = more utility, less privacy
        self.min_weight  = min_weight
        self.rng         = np.random.default_rng(random_state)

    def _mi_column_weights(self, V: np.ndarray) -> np.ndarray:
        from sklearn.feature_selection import mutual_info_classif
        V_tr    = V[:self.n_train]
        mi      = mutual_info_classif(V_tr, self.y_train,
                      discrete_features=False, random_state=42)
        mi      = np.maximum(mi, self.min_weight * (mi.max() + 1e-10))
        return mi / (mi.sum() + 1e-10)

    def perturb(self, V: np.ndarray) -> tuple:
        n, k = V.shape

        # ── Per-column stats anchored to actual V scale ────────────────
        col_std  = V.std(axis=0)          # (k,)  typically O(1/√n)

        # ── MI weights ────────────────────────────────────────────────
        weights  = self._mi_column_weights(V)

        # ── Laplace scales anchored to col_std ────────────────────────
        # scale_i = (w_i * col_std_i * k) / (epsilon * snr_target)
        # snr_target controls the signal/noise tradeoff directly:
        #   snr=1 → noise_std ≈ signal_std  (strong privacy, low utility)
        #   snr=5 → noise_std ≈ 0.2×signal  (mild privacy, good utility)
        scales        = (weights * col_std * k) / (self.epsilon * self.snr_target)
        laplace_noise = np.zeros((n, k))
        for i in range(k):
            laplace_noise[:, i] = self.rng.laplace(0.0, scales[i], size=n)

        # ── Gaussian floor anchored to col_std ────────────────────────
        # sigma = (min_weight * col_std.mean()) / (epsilon * snr_target)
        # This ensures the floor never overwhelms signal
        sigma_floor   = (self.min_weight * col_std.mean()) / (self.epsilon * self.snr_target)
        gaussian_noise = self.rng.normal(0.0, sigma_floor, size=(n, k))

        V_pert = V + laplace_noise + gaussian_noise

        # Diagnostics — the key ones to watch
        actual_snr = col_std / (scales * np.sqrt(2) + sigma_floor)   # approx SNR per col

        meta = {
            "epsilon"        : self.epsilon,
            "snr_target"     : self.snr_target,
            "col_std"        : col_std.tolist(),
            "laplace_scales" : scales.tolist(),
            "gaussian_sigma" : round(float(sigma_floor), 8),
            "actual_snr"     : actual_snr.tolist(),    # watch this — should be > 1
            "mi_weights"     : weights.tolist(),
            "frob_ratio"     : round(
                float(np.linalg.norm(V_pert, 'fro')) /
                float(np.linalg.norm(V, 'fro')), 4),
        }
        return V_pert, meta

In [20]:
model_results = run_classification(features_clean_embed, y, target_names=target_names)
display_classification_results(model_results)
run_cv_inference_attack(features_clean_embed, y, sensitive_class=sensitive_label)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
1,Random Forest,0.821,0.830,0.832,0.831
3,KNN,0.799,0.808,0.817,0.812
4,Decision Tree,0.793,0.803,0.803,0.803
2,SVM,0.668,0.816,0.626,0.629
0,Logistic Regression,0.509,0.359,0.421,0.355



  Inference Attack
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  Status
  --------------------------------------------------------------
  Logistic Regression (LR)   0.7840 +0.0000 0.0000 0.7150  ✗ leaked
  Random Forest (RF)         0.9211 -0.1371 0.6346 0.9563  ✗ leaked



,Adv Accuracy,Random Baseline,Privacy Gain,Norm Gain,Adv F1,Adv AUC
Logistic Regression (LR),0.7840,0.784,0.0000,0.0000,0.0000,0.7150
Random Forest (RF),0.9211,0.784,-0.1371,0.6346,0.8147,0.9563


In [33]:
snr_values = [0.5, 1.0, 2.0, 5.0, 10.0]
rows = []

for snr in snr_values:
    ep = EmbeddingPerturbation(
        epsilon     = 1.0,
        y_train     = y,
        n_train     = len(y),
        snr_target  = snr,
        min_weight  = 0.05,
        random_state= 42,
    )
    V_pert, meta = ep.perturb(features_clean_embed)

    # ── Classification utility ─────────────────────────────────────────
    model_results = run_classification(V_pert, y, target_names=target_names)
    macro_f1      = model_results["Random Forest"]["macro_f1"]

    # ── Inference attack ───────────────────────────────────────────────
    attack_df = run_cv_inference_attack(
        V_pert, y,
        sensitive_class = sensitive_label,
        stage_name      = f"SNR={snr}",
    )
    adv_acc   = float(attack_df.loc["Logistic Regression (LR)", "Adv Accuracy"])
    adv_auc   = float(attack_df.loc["Logistic Regression (LR)", "Adv AUC"])
    baseline  = float(attack_df.loc["Logistic Regression (LR)", "Random Baseline"])
    norm_gain = float(attack_df.loc["Logistic Regression (LR)", "Norm Gain"])

    rows.append({
        "SNR target"  : snr,
        "Frob ratio"  : round(meta["frob_ratio"], 3),
        "Actual SNR"  : [round(s, 2) for s in meta["actual_snr"]],
        "Macro F1"    : round(macro_f1,  4),
        "Adv Accuracy": round(adv_acc,   4),
        "Adv AUC"     : round(adv_auc,   4),
        "Norm Gain"   : round(norm_gain, 4),
        "Privacy Gain": round(baseline - adv_acc, 4),
    })

    print(
        f"SNR={snr:<5}  F1={macro_f1:.3f}  "
        f"adv={adv_acc:.3f}  AUC={adv_auc:.3f}  "
        f"norm_gain={norm_gain:+.3f}  "
        f"{'✓' if norm_gain < 0 else '✗'}"
    )

df_sweep = pd.DataFrame(rows)

# ── Clean embedding reference ──────────────────────────────────────────
clean_results   = run_classification(features_clean_embed, y, target_names=target_names)
f1_clean        = clean_results["Random Forest"]["macro_f1"]

attack_df_clean = run_cv_inference_attack(
    features_clean_embed, y,
    sensitive_class = sensitive_label,
    stage_name      = "Clean Embedding (Reference)",
)
adv_acc_c   = float(attack_df_clean.loc["Logistic Regression (LR)", "Adv Accuracy"])
adv_auc_c   = float(attack_df_clean.loc["Logistic Regression (LR)", "Adv AUC"])
baseline_c  = float(attack_df_clean.loc["Logistic Regression (LR)", "Random Baseline"])
norm_gain_c = float(attack_df_clean.loc["Logistic Regression (LR)", "Norm Gain"])

reference = pd.DataFrame([{
    "SNR target"  : "clean (ref)",
    "Frob ratio"  : 1.0,
    "Actual SNR"  : "—",
    "Macro F1"    : round(f1_clean,    4),
    "Adv Accuracy": round(adv_acc_c,   4),
    "Adv AUC"     : round(adv_auc_c,   4),
    "Norm Gain"   : round(norm_gain_c, 4),
    "Privacy Gain": round(baseline_c - adv_acc_c, 4),
}])

df_full = pd.concat([reference, df_sweep], ignore_index=True)
print("\n", df_full.to_string(index=False))


  Inference Attack — SNR=0.5
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  Status
  --------------------------------------------------------------
  Logistic Regression (LR)   0.5806 +0.2034 -0.9414 0.6111  ✓ private

SNR=0.5    F1=0.368  adv=0.581  AUC=0.611  norm_gain=-0.941  ✓

  Inference Attack — SNR=1.0
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  Status
  --------------------------------------------------------------
  Logistic Regression (LR)   0.6050 +0.1790 -0.8284 0.6643  ✓ private

SNR=1.0    F1=0.472  adv=0.605  AUC=0.664  norm_gain=-0.828  ✓

  Inference Attack — SNR=2.0
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  

In [21]:
# ── After fit_transform ───────────────────────────────────────────────
ep = EmbeddingPerturbation(
    epsilon      = 0.5,
    y_train      = y,          # training split only
    n_train      = len(y),
    delta        = 1e-5,
    alpha        = 1.0,              # amplifier — increase if attack still wins
    min_weight   = 0.05,
    random_state = 42,
)

V_ep, ep_meta = ep.perturb(features_clean_embed)

print(f"MI weights per column : {[round(w,3) for w in ep_meta['mi_weights']]}")
print(f"Laplace scales        : {[round(s,4) for s in ep_meta['laplace_scales']]}")
print(f"Gaussian sigma        : {ep_meta['gaussian_sigma']:.6f}")
print(f"Frob ratio            : {ep_meta['V_frob_after']/ep_meta['V_frob_before']:.4f}")

# Stage 5: run classification + attack on V_ep
clf_ep = run_classification(V_ep, y, target_names)
atk_ep = run_cv_inference_attack(V_ep, y, sensitive_class=sensitive_label, stage_name="Embedding Perturbation")

MI weights per column : [0.263, 0.288, 0.449]
Laplace scales        : [1.5772, 1.7294, 2.6934]
Gaussian sigma        : 0.000047
Frob ratio            : 276.2824

  Inference Attack — Embedding Perturbation
  Sensitive class : 0
  Class balance   : 21.6% positive  /  78.4% negative
  Random baseline : 0.7840

  Adversary                     Acc     Gain   NormG     AUC  Status
  --------------------------------------------------------------
  Logistic Regression (LR)   0.7840 +0.0000 0.0000 0.5010  ✗ leaked
  Random Forest (RF)         0.7732 +0.0107 -0.0496 0.4932  ✓ private



In [22]:
display_classification_results(clf_ep)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
4,Decision Tree,0.271,0.249,0.249,0.249
3,KNN,0.263,0.240,0.240,0.239
1,Random Forest,0.301,0.237,0.242,0.229
2,SVM,0.378,0.213,0.250,0.140
0,Logistic Regression,0.379,0.095,0.250,0.138
